In [1]:
# =====================================================
# 🏆 MODE PIPELINE + OPTUNA TUNING (v4.0)
# =====================================================

import numpy as np
import pandas as pd
import shap
import gc
import warnings
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance

# Model Kütüphaneleri
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

In [2]:
# =====================================================
# 📥 DATA LOAD & PREP
# =====================================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = np.log1p(train['target'])
train_ids = train['id']
test_ids = test['id']

In [3]:
# =====================================================
# 🔥 SMOOTHED TARGET ENCODING
# =====================================================
def smoothed_target_encoding(train_df, test_df, column, target, alpha=10):
    global_mean = target.mean()
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    train_df[f'{column}_te'] = 0
    
    for tr_idx, val_idx in kf.split(train_df):
        tr_fold = train_df.iloc[tr_idx]
        y_fold = target.iloc[tr_idx]
        agg = pd.concat([tr_fold[column], y_fold], axis=1).groupby(column)[target.name].agg(['count', 'mean'])
        smooth = (agg['count'] * agg['mean'] + alpha * global_mean) / (agg['count'] + alpha)
        train_df.loc[val_idx, f'{column}_te'] = train_df.iloc[val_idx][column].map(smooth)
    
    agg = pd.concat([train_df[column], target], axis=1).groupby(column)[target.name].agg(['count', 'mean'])
    smooth = (agg['count'] * agg['mean'] + alpha * global_mean) / (agg['count'] + alpha)
    test_df[f'{column}_te'] = test_df[column].map(smooth)
    
    train_df[f'{column}_te'].fillna(global_mean, inplace=True)
    test_df[f'{column}_te'].fillna(global_mean, inplace=True)
    return train_df, test_df

train, test = smoothed_target_encoding(train, test, 'stock_id', pd.Series(y, name='target'))

drop_cols = ['id', 'target', 'stock_id']
X = train.drop(columns=[c for c in drop_cols if c in train.columns])
X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

In [4]:
# =====================================================
# 🧠 HYBRID FEATURE SELECTION
# =====================================================
print("🔍 Feature Selection in progress...")
fs_model = XGBRegressor(n_estimators=200, max_depth=5, tree_method="hist", device="cpu")
fs_model.fit(X, y)

explainer = shap.Explainer(fs_model)
shap_values = explainer(X)
shap_imp = np.abs(shap_values.values).mean(axis=0)

perm = permutation_importance(fs_model, X, y, n_repeats=2, random_state=42)

feat_df = pd.DataFrame({
    "feature": X.columns,
    "score": (shap_imp * 0.6) + (perm.importances_mean * 0.4)
}).sort_values("score", ascending=False)

selected = feat_df.head(int(len(feat_df) * 0.70))['feature'].tolist()
X, X_test = X[selected], X_test[selected]
print(f"✅ Selected {len(selected)} high-impact features.")

🔍 Feature Selection in progress...
✅ Selected 19 high-impact features.


In [5]:
# =====================================================
# ⚡ OPTUNA TUNING
# =====================================================
def objective(trial, model_name, X, y):
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    if model_name == "xgb":
        # ... (xgb parametreleri aynı kalabilir) ...
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1000, 4000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
            "tree_method": "hist",
            "device": "cpu"
        }
        model_class = XGBRegressor

    elif model_name == "lgb":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 1000, 4000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
            "num_leaves": trial.suggest_int("num_leaves", 31, 45),
            "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
            "n_jobs": -1,
            "verbosity": -1 # <--- Buraya verbosity ekleyerek logları kapatabilirsin
        }
        model_class = LGBMRegressor

    elif model_name == "cat":
        # ... (cat parametreleri aynı kalabilir) ...
        params = {
            "iterations": trial.suggest_int("iterations", 1000, 4000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
            "depth": trial.suggest_int("depth", 4, 7),
            "early_stopping_rounds": 100,
            "verbose": 0
        }
        model_class = CatBoostRegressor

    for tr_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = model_class(**params)
        
        # --- DÜZELTİLMİŞ FİT SATIRI ---
        if model_name == "lgb":
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                      callbacks=[log_evaluation(period=0)]) # verbose=0 yerine bu
        else:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
            
        oof[val_idx] = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y, oof))
    return rmse
    
print("🔧 Running Optuna tuning...")

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(lambda trial: objective(trial, "xgb", X, y), n_trials=20)

study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(lambda trial: objective(trial, "lgb", X, y), n_trials=20)

study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(lambda trial: objective(trial, "cat", X, y), n_trials=20)

best_xgb = study_xgb.best_params
best_lgb = study_lgb.best_params
best_cat = study_cat.best_params

print("✅ Best XGB params:", best_xgb)
print("✅ Best LGB params:", best_lgb)
print("✅ Best CAT params:", best_cat)

[I 2026-04-20 22:48:09,760] A new study created in memory with name: no-name-7240c62f-f923-4ef8-9fe1-7e0fc1ee9f90


🔧 Running Optuna tuning...


[I 2026-04-20 22:49:13,705] Trial 0 finished with value: 0.3452663434407612 and parameters: {'n_estimators': 1070, 'learning_rate': 0.016764180363011447, 'max_depth': 4, 'subsample': 0.8282667551986784, 'colsample_bytree': 0.876071822882674}. Best is trial 0 with value: 0.3452663434407612.
[I 2026-04-20 22:51:49,718] Trial 1 finished with value: 0.34631074385571453 and parameters: {'n_estimators': 2211, 'learning_rate': 0.04983414018019781, 'max_depth': 5, 'subsample': 0.8996926054589054, 'colsample_bytree': 0.6187077053208504}. Best is trial 0 with value: 0.3452663434407612.
[I 2026-04-20 22:53:10,561] Trial 2 finished with value: 0.3459700452593145 and parameters: {'n_estimators': 1039, 'learning_rate': 0.04751441460732876, 'max_depth': 6, 'subsample': 0.7955621141745489, 'colsample_bytree': 0.8113360151723081}. Best is trial 0 with value: 0.3452663434407612.
[I 2026-04-20 22:56:09,365] Trial 3 finished with value: 0.34548061727217905 and parameters: {'n_estimators': 3024, 'learning_

✅ Best XGB params: {'n_estimators': 1729, 'learning_rate': 0.010425817309932476, 'max_depth': 5, 'subsample': 0.7667510738879703, 'colsample_bytree': 0.732453518189379}
✅ Best LGB params: {'n_estimators': 2249, 'learning_rate': 0.010159794383574845, 'num_leaves': 34, 'subsample': 0.6233838118789714, 'colsample_bytree': 0.7160595928022824}
✅ Best CAT params: {'iterations': 1668, 'learning_rate': 0.031210167782555386, 'depth': 7}


In [6]:
# =====================================================
# 🚀 ENSEMBLE TRAINING (XGB, LGB, CAT with tuned params)
# =====================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n🚀 FOLD {fold+1} STARTING...")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    xgb = XGBRegressor(**best_xgb)
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    oof_preds[val_idx, 0] = xgb.predict(X_val)
    test_preds[:, 0] += xgb.predict(X_test) / kf.n_splits

    lgb = LGBMRegressor(**best_lgb)
    lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[early_stopping(100), log_evaluation(0)])
    oof_preds[val_idx, 1] = lgb.predict(X_val)
    test_preds[:, 1] += lgb.predict(X_test) / kf.n_splits

    cat = CatBoostRegressor(**best_cat)
    cat.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    oof_preds[val_idx, 2] = cat.predict(X_val)
    test_preds[:, 2] += cat.predict(X_test) / kf.n_splits

    del X_tr, X_val; gc.collect()

# =====================================================


🚀 FOLD 1 STARTING...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1285]	valid_0's l2: 0.119108
0:	learn: 0.3465143	test: 0.3465263	best: 0.3465263 (0)	total: 51.6ms	remaining: 1m 26s
1:	learn: 0.3464588	test: 0.3464697	best: 0.3464697 (1)	total: 103ms	remaining: 1m 25s
2:	learn: 0.3464141	test: 0.3464236	best: 0.3464236 (2)	total: 156ms	remaining: 1m 26s
3:	learn: 0.3463698	test: 0.3463800	best: 0.3463800 (3)	total: 209ms	remaining: 1m 26s
4:	learn: 0.3463253	test: 0.3463364	best: 0.3463364 (4)	total: 261ms	remaining: 1m 26s
5:	learn: 0.3462848	test: 0.3462948	best: 0.3462948 (5)	total: 315ms	remaining: 1m 27s
6:	learn: 0.3462435	test: 0.3462562	best: 0.3462562 (6)	total: 369ms	remaining: 1m 27s
7:	learn: 0.3462073	test: 0.3462221	best: 0.3462221 (7)	total: 421ms	remaining: 1m 27s
8:	learn: 0.3461717	test: 0.3461872	best: 0.3461872 (8)	total: 473ms	remaining: 1m 27s
9:	learn: 0.3461374	test: 0.3461550	best: 0.3461550 (9)	total: 527m

In [7]:
# =====================================================
# 🛠️ STACKING & FINAL SUBMISSION
# =====================================================
meta_learner = Ridge(alpha=1.0)
meta_learner.fit(oof_preds, y)

weights = meta_learner.coef_
print(f"\n📊 Meta-Model Weights -> XGB: {weights[0]:.3f}, LGB: {weights[1]:.3f}, CAT: {weights[2]:.3f}")

final_oof_rmse = np.sqrt(mean_squared_error(y, meta_learner.predict(oof_preds)))
print(f"⭐ Final Stacking RMSE: {final_oof_rmse:.6f}")

# Test tahminleri ve log dönüşümü
final_test_preds = np.expm1(meta_learner.predict(test_preds))

# Submission dosyası
submission = pd.DataFrame({"id": test_ids, "target": final_test_preds})
submission.to_csv("god_mode_optuna.csv", index=False)
print("\n🏆 MISSION ACCOMPLISHED: V2.csv created!")



📊 Meta-Model Weights -> XGB: -0.015, LGB: 0.296, CAT: 0.750
⭐ Final Stacking RMSE: 0.345082

🏆 MISSION ACCOMPLISHED: V2.csv created!
